# Moving Average vs Naive Forecast Evaluation

This notebook provides a statistical comparison between the 3-point moving average forecasting method and a naive last-value baseline across synthetic noisy time series samples. We compute Mean Squared Error (MSE), Root Mean Squared Error (RMSE), paired t-test, and Wilcoxon signed-rank test statistics to rigorously assess statistical significance and robustness.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2')

## Imports & Setup

Import necessary libraries for numerical computation, statistical testing, and plotting.

In [ ]:
import json
import os
import urllib.request
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

## Data Loading Helper

Load data from GitHub raw URL with local fallback.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-230d6e-robust-temporal-smoothing-evaluating-mov/main/round-1/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded dataset with {len(data['datasets'][0]['examples'])} examples.")

## Configuration

Define configuration parameters for evaluation.

In [ ]:
# Configuration parameters
CONFIG = {
    "max_samples": 100  # limit number of samples to process in demo
}

## Processing & Evaluation

Extract actuals, naive predictions, and moving average predictions from the dataset, then compute error metrics and statistical tests.

In [ ]:
examples = data["datasets"][0]["examples"][:CONFIG["max_samples"]]

actuals = []
naive_preds = []
ma_preds = []

for ex in examples:
    actuals.append(float(ex["output"]))
    naive_preds.append(float(ex["predict_naive"]))
    ma_preds.append(float(ex["predict_moving_average"]))

actuals = np.array(actuals)
naive_preds = np.array(naive_preds)
ma_preds = np.array(ma_preds)

naive_errors = (naive_preds - actuals) ** 2
ma_errors = (ma_preds - actuals) ** 2

naive_mse = float(np.mean(naive_errors))
ma_mse = float(np.mean(ma_errors))

naive_rmse = float(np.sqrt(naive_mse))
ma_rmse = float(np.sqrt(ma_mse))

# Paired t-test and Wilcoxon signed-rank test on squared errors
t_stat, p_value_t = stats.ttest_rel(naive_errors, ma_errors)
wilcoxon_stat, p_value_w = stats.wilcoxon(naive_errors, ma_errors)

metrics = {
    "naive_mse": naive_mse,
    "moving_average_mse": ma_mse,
    "naive_rmse": naive_rmse,
    "moving_average_rmse": ma_rmse,
    "paired_t_statistic": float(t_stat),
    "paired_t_p_value": float(p_value_t),
    "wilcoxon_statistic": float(wilcoxon_stat),
    "wilcoxon_p_value": float(p_value_w),
    "num_samples": len(actuals)
}

print("Evaluation Metrics:")
print(json.dumps(metrics, indent=2))

## Results Visualization & Summary

Visualize the comparison of MSE and RMSE between the naive baseline and the 3-point moving average method.

In [ ]:
labels = ['Naive Forecast', '3-Point Moving Average']
mse_values = [metrics["naive_mse"], metrics["moving_average_mse"]]
rmse_values = [metrics["naive_rmse"], metrics["moving_average_rmse"]]

x = np.arange(len(labels))
width = 0.35

fig, ax1 = plt.subplots(figsize=(8, 5))

rects1 = ax1.bar(x - width/2, mse_values, width, label='MSE', color='skyblue')
rects2 = ax1.bar(x + width/2, rmse_values, width, label='RMSE', color='salmon')

ax1.set_ylabel('Error Value')
ax1.set_title('Comparison of Naive vs Moving Average Forecast Errors')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.legend()

fig.tight_layout()
plt.show()